In [1]:
import argparse
from os import path
import pandas as pd

import mne
import numpy as np
from scipy.stats import permutation_test, false_discovery_control, norm
from scipy.signal import detrend
from mne_bids import BIDSPath, read_raw_bids
from mne_bids.dig import _read_dig_bids
from mne_bids.path import _find_matching_sidecar
from ieeg.navigate import channel_outlier_marker
from ieeg.navigate import outliers_to_nan
from ieeg.calc.scaling import rescale
from ieeg.timefreq import gamma
from ieeg.timefreq.multitaper import spectrogram
from ieeg.timefreq.utils import crop_pad
from ieeg.viz.ensemble import plot_chan_map
from ieeg.io import raw_from_layout
from bids.layout import BIDSLayout
from pathlib import Path
from ieeg.navigate import trial_ieeg
from mne_bids import get_bids_path_from_fname
import re
import logging
import sys
from joblib import Parallel, delayed

Qt5Agg backend not available, using default backend
Qt5Agg backend not available, using default backend


In [2]:
def find_channel_outliers(data, threshold=3):
    data = detrend(data, axis=1)  # ensure detrending along time
    R2 = np.square(data)
    R2[R2 == 0] = 1e-9

    sig = np.std(R2, axis=1)
    m, s = norm.fit(sig)
    out1 = np.where(sig > (threshold*s + m))[0]
    mask = np.setdiff1d(np.arange(len(sig)), out1)
    m2, s2 = norm.fit(sig[mask])
    out2 = np.where(sig[mask] > (threshold*s2 + m2))[0]
    chan_outliers = np.sort(np.concatenate([out1, mask[out2]])) + 1
    chan_outliers = [str(chan) for chan in chan_outliers]
    return chan_outliers

def remove_bad_trials(data, threshold=10, method=1):
    """Removes bad trials based on a threshold detection.

    This function evaluates each trial for every channel and removes those
    considered "bad" based on the specified thresholding method.

    Args:
        data (np.ndarray): 
            Array of shape (electrodes, trials, samples) representing the recorded data.
        threshold (float, optional): 
            Threshold for identifying noisy trials. Defaults to 10.
        method (int, optional): 
            Method for thresholding:
            
            - 1: Uses an absolute amplitude threshold based on mean and standard deviation.
            - 2: Uses a differential threshold on the first temporal derivative.
            
            Defaults to 1.

    Returns:
        tuple[np.ndarray, np.ndarray]: 
            - **NumTrials** (`np.ndarray`): Array of shape (electrodes,) containing 
              the number of good trials for each channel.
            - **goodtrials** (`np.ndarray`): Boolean array of shape (electrodes, trials)
              indicating which trials are considered good.

    Raises:
        ValueError: If `method` is not 1 or 2.
    """
    n_trials, n_channels, _ = data.shape
    NumTrials = np.zeros(n_channels, dtype=int)
    goodtrials = np.zeros((n_channels, n_trials), dtype=bool)

    for iCh in range(n_channels):
        tmp = data[:, iCh, :]  # shape: (trials, samples)

        if method == 1:
            th = threshold * np.std(np.abs(tmp)) + np.mean(np.abs(tmp))
            e = np.max(np.abs(tmp), axis=1)
            mask = (e < th) & (e != 0)
        elif method == 2:
            difftmp = np.diff(tmp, axis=1)
            e = np.max(difftmp, axis=1)
            mask = e < threshold
        else:
            raise ValueError("method must be 1 or 2")

        NumTrials[iCh] = np.sum(mask)
        goodtrials[iCh, :] = mask

    return NumTrials, goodtrials

In [8]:
# Simple logging: everything INFO and above to stdout
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)

def preproc(
    bids_layout: BIDSLayout,
    subject: str,
    tmin: float,
    tmax: float,
    fs: int,
    ref: str,
    **kwargs
):
    # Load data
    raw = raw_from_layout(
        bids_layout,
        subject=subject,
        extension='.edf',
        desc=None,
        **kwargs
    )
    raw.load_data()
    raw._data *= 1e6  # convert to uV
    nChans = raw.info['nchan']

    events, event_id = mne.events_from_annotations(raw)
    baseline_events = [e for e in event_id.keys() if 'stimulus' in e]
    resp_events = [e for e in event_id.keys() if 'response' in e]

    resp_epochs = trial_ieeg(
        raw,
        event=resp_events,
        times=[-3, 3],
        preload=True,
    )
    # channel outlier removal based on detrended concatenated data
    data_chan_time = resp_epochs.get_data().transpose(1,0,2).reshape(resp_epochs.info['nchan'], -1)
    bad_channels_detrend = find_channel_outliers(data_chan_time)
    print(f"IEEG BAD CHANNELS DETREND PHONEME EPOCHS: {bad_channels_detrend}")
    bad_channels_tot = list(set(raw.info['bads'] + bad_channels_detrend))

    # update bads in raw objects with new outliers
    # raw.info['bads'] = bad_channels_tot
    # raw_phon.info['bads'] = bad_channels_tot
    raw.drop_channels(bad_channels_tot)

    # remove Trigger channel we have any
    raw.drop_channels(['Trigger']) if 'Trigger' in raw.ch_names else None
    ch_type = raw.get_channel_types(only_data_chs=True)[0]
    print(f"Available channel types: {ch_type}")

    # Notch filter power line noise
    powerline_freq = raw.info.get("line_freq")
    if powerline_freq is None:
        powerline_freq = 60
    freqs = [powerline_freq * m for m in range(1, 4)]
    raw.notch_filter(freqs=freqs, notch_widths=2, n_jobs=-1)

    # Re-reference data
    logger.info(f'Setting re-reference scheme: {ref}')

    raw = raw.set_eeg_reference(ref_channels="average", ch_type=ch_type)


    base_epochs = trial_ieeg(
        raw,
        event=baseline_events,
        times=[-1, 0.5],
        preload=True,
    )
    # recreate raw after filtering and CAR
    resp_epochs = trial_ieeg(
        raw,
        event=resp_events,
        times=[-3, 3],
        preload=True,
    )

    ##### Bad Trial Removal #####
    tr_remove_data_prod = resp_epochs._data
    _, goodtrials_production = remove_bad_trials(tr_remove_data_prod, threshold=10, method=1)
    nChans = tr_remove_data_prod.shape[1]
    goodtrials_production_common = np.sum(goodtrials_production, axis=0) >= (0.8 * nChans)
    resp_epochs = resp_epochs[goodtrials_production_common]
    base_epochs = base_epochs[goodtrials_production_common]
    ##########

    freq = np.arange(10, 200., 6.)
    spectra = spectrogram(resp_epochs, freq, base_epochs, verbose=10, n_cycles=freq/2)
    crop_pad(spectra, "0.5s")

    print('test')

    # plot_chan_map(spectra, )


def remove_arpabet_stress(phoneme: str) -> str:
    """Remove stress markers from ARPAbet phonemes.

    E.g., 'AH0' -> 'AH', 'IY1' -> 'IY'
    """
    return re.sub(r'\d', '', phoneme)


def swap_kv_dict(d):
    return dict((v, k) for k, v in d.items())

def main(bids_root: str, subject: str, tmin: float = -1.0,
         tmax: float = 1.5, fs: int = 200, **kwargs):

    bids_layout = BIDSLayout(
        root=bids_root,
        derivatives=True,
    )
    preproc(
        bids_layout=bids_layout,
        subject=subject,
        tmin=tmin,
        tmax=tmax,
        fs=fs,
        **kwargs
    )


In [9]:
main(bids_root='/hpc/home/zms14/cworkspace/BIDS_1.0_Lexical_µECoG/BIDS', subject='S81', tmin=-1, tmax=1.5, fs=200, ref='CAR')

Extracting EDF parameters from /hpc/home/zms14/cworkspace/BIDS_1.0_Lexical_µECoG/BIDS/sub-S81/ieeg/sub-S81_task-lexical_ieeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading events from /hpc/home/zms14/cworkspace/BIDS_1.0_Lexical_µECoG/BIDS/sub-S81/ieeg/sub-S81_task-lexical_events.tsv.
The event "response" refers to multiple event values.Creating hierarchical event names.
    Renaming event: response -> response/relic
    Renaming event: response -> response/halet
    Renaming event: response -> response/delin
    Renaming event: response -> response/vanic
    Renaming event: response -> response/civil
    Renaming event: response -> response/bagel
    Renaming event: response -> response/basin
    Renaming event: response -> response/comic
    Renaming event: response -> response/tacin
    Renaming event: response -> response/viral
    Renaming event: response -> response/limid
    Renaming event: response -> response/topic
    Renaming

: 